In [11]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [12]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법_with_table.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 6.61 s
Wall time: 6.68 s


180

In [13]:
documents[43]

Document(metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천만원 초과 3억원 이하\n\n3,706만원 + (1억5천만원을 초과하는 금액의

In [15]:
# 최초 1회, 현재 노트북 커널에 패키지가 없을 때 아래 줄의 #을 지우고 실행하세요.
# %pip install -U langchain-pinecone python-dotenv

from dotenv import load_dotenv
from langchain_pinecone import PineconeEmbeddings

load_dotenv()  # .env의 PINECONE_API_KEY 사용 (Upstage 키가 아님)

# Pinecone 화면의 모델과 차원(1024)에 맞춥니다.
# 문서는 passage, 질문은 query 용도로 같은 모델을 사용합니다.
embedding = PineconeEmbeddings(
    model="llama-text-embed-v2",
    dimension=1024,
    query_params={"input_type": "query", "dimension": 1024, "truncate": "END"},
    document_params={"input_type": "passage", "dimension": 1024, "truncate": "END"},
)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.10.0 requires protobuf<3.20,>=3.9.2, but you have protobuf 7.36.2 which is incompatible.
transformers 4.57.3 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


   ---------------------------------------- 0.0/587.6 kB ? eta -:--:--
   ---------------------------------------- 587.6/587.6 kB 10.5 MB/s  0:00:00

  Attempting uninstall: packaging

    Found existing installation: packaging 26.3

    Uninstalling packaging-26.3:

      Successfully uninstalled packaging-26.3

   ------------- -------------------------- 2/6 [packaging]
   -------------------- ------------------- 3/6 [pinecone-plugin-assistant]
   -------------------- ------------------- 3/6 [pinecone-plugin-assistant]
   -------------------- ------------------- 3/6 [pinecone-plugin-assistant]
   -------------------------- ------------- 4/6 [pinecone]
   -------------------------- ------------- 4/6 [pinecone]
   -------------------------- ------------- 4/6 [pinecone]
   -------------------------- ------------- 4/6 [pinecone]
   -------------------------- ------------- 4/6 [pinecone]
   -------------------------- ------------- 4/6 [pinecone]
   -------------------------- -------------

In [16]:
# 실제 Pinecone 임베딩 API를 호출하는 확인 셀입니다.
query_vector = embedding.embed_query("소득세법")
dimension = len(query_vector)
assert dimension == 1024, f"임베딩 차원이 다릅니다: {dimension}"
print("질문 벡터 차원:", dimension)


질문 벡터 차원: 1024


In [20]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec

# 0. pinecone 객체 생성
pc = Pinecone()  # .env의 PINECONE_API_KEY 사용

# 1. 인덱스 설정
index_name = 'tax-index'
dimension = len(embedding.embed_query("차원 확인용"))

# 2. 인덱스 생성(해당 인덱스가 없을 경우)
existing_indexes = [index.name for index in pc.list_indexes()]
# print(index_name in existing_indexes)

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric='cosine',  # 유사도 측정 방식
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

    # 인덱스가 준비될 때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)  # 1초 대기

    print(f'인덱스 {index_name}를 생성하였습니다')
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')

인덱스 tax-index이 이미 존재합니다


In [22]:
# 3. 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])

['tax-index']
True


In [26]:
%%time
# 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
import time
from pinecone.exceptions import PineconeApiException
from langchain_pinecone import PineconeVectorStore

database = PineconeVectorStore(
    index=pc.Index(index_name),
    embedding=embedding,
    namespace='tax-table-llama-v2'
)

batch_size = 10

for start in range(0, len(documents), batch_size):
    batch = documents[start:start + batch_size]

    # 같은 문서 목록으로 재실행할 때 중복 저장 방지
    ids = [f'tax-table-{i}' for i in range(start, start + len(batch))]

    for attempt in range(5):
        try:
            database.add_documents(
                documents=batch,
                ids=ids
            )
            break

        except PineconeApiException as e:
            if e.status != 429 or attempt == 4:
                raise

            wait_seconds = 60 * (attempt + 1)
            print(f'요청 제한: {wait_seconds}초 후 재시도합니다.')
            time.sleep(wait_seconds)

    print(f'{start + len(batch)} / {len(documents)}개 저장 완료')

    if start + len(batch) < len(documents):
        time.sleep(15)

Index host ignored when initializing with index object.


10 / 180개 저장 완료
20 / 180개 저장 완료
30 / 180개 저장 완료
40 / 180개 저장 완료
50 / 180개 저장 완료
60 / 180개 저장 완료
70 / 180개 저장 완료
80 / 180개 저장 완료
90 / 180개 저장 완료
100 / 180개 저장 완료
110 / 180개 저장 완료
120 / 180개 저장 완료
130 / 180개 저장 완료
140 / 180개 저장 완료
150 / 180개 저장 완료
160 / 180개 저장 완료
170 / 180개 저장 완료
180 / 180개 저장 완료
CPU times: total: 2.77 s
Wall time: 4min 39s


# 2. 답변 생성 전 retrieval 확인

In [27]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [28]:
retrieved_docs[0]

Document(id='tax-table-43', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천만원 초과 3억원 이하\n\n3,706만원 

In [30]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])
# print(retrieved_doc)

In [31]:
'55조' in retrieved_doc

True

# 3. 답변 생성

In [33]:
from openai import OpenAI
client = OpenAI()
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages =[
        {'role' : 'system', 'content' : '당신은 최고의 한국 소득세법 전문가입니다.'},
        {
            'role' : 'user',
            'content' : f"""- [context]를 참고해서 사용자의 질문에 10줄 이내로 답변해주세요.
            - [context] : {retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)

In [36]:
result = response.choices[0].message.content
print(result)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 종합소득세율을 적용해야 합니다. 

1. 5천만원의 경우, 과세표준은 5천만원 초과 8,800만원 이하에 해당하므로, 세율은 24%입니다.
2. 계산식은 다음과 같습니다:
   - 5,000만원 초과분: 5,000만원 초과분은 5,000만원에서 1,400만원을 초과하는 금액에 대해 24% 적용.
   - 기본 세액: 624만원 + (5,000만원 - 1,400만원) × 24% = 624만원 + 840만원 = 1,464만원.

따라서, 연봉 5천만원인 직장인의 소득세는 약 1,464만원입니다. 단, 기부금, 세액공제 등 개인의 상황에 따라 달라질 수 있습니다.


In [37]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-mini')

In [38]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model = 'solar-pro4',
    reasoning_effort='low' # 느리지만 더 깊게 추론하여 답변 (low, medium, high)
)

In [39]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고하여 사용자의 질문에 답변해주세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [40]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉 5,000만 원인 직장인의 소득세는 **각종 공제에 따라 달라지지만**, 가장 기본적인 공제만 적용한 사례와 함께 실제 세액을 계산해드리겠습니다.  
주어진 소득세법 제55조 세율표를 기준으로 설명합니다.

---

### 1. 소득공제 후 과세표준 추정 (1인 가구, 부양가족 없음, 다른 소득공제 없음 가정)

**(1) 근로소득공제**  
소득세법 제47조에 따른 근로소득공제액 계산  
- 총급여 5,000만 원 → 4,500만 원 초과 ~ 1억 원 이하 구간  
- 공제액 = 1,200만 원 + (5,000만 원 - 4,500만 원) × 5%  
        = 1,200만 원 + 25만 원 = **1,225만 원**

→ 근로소득금액 = 5,000만 원 - 1,225만 원 = **3,775만 원**

**(2) 기본공제 (본인 1인)**  
- 기본공제 1인당 150만 원  
→ 과세표준 = 3,775만 원 - 150만 원 = **3,625만 원**

(여기에 연금보험료 공제, 신용카드 공제, 보험료·의료비·교육비 공제 등이 추가될 수 있지만, 없는 것으로 가정)

---

### 2. 산출세액 계산 (과세표준 3,625만 원, 제55조 세율 적용)

제55조 세율표에서 3,625만 원은 “1,400만 원 초과 ~ 5,000만 원 이하” 구간입니다.

**산출세액**  
= 84만 원 + (3,625만 원 - 1,400만 원) × 15%  
= 84만 원 + 2,225만 원 × 0.15  
= 84만 원 + 333.75만 원  
= **4,177,500원**

---

### 3. 세액공제 (표준세액공제 적용)

해당 직장인이 의료비·보험료·기부금 등 특별세액공제를 신청하지 않은 **근로소득자**라면,  
제59조의4(표준세액공제)에 따라 **연 13만 원**을 종합소득산출세액에서 공제합니다.  
(※ 근로소득세액공제는 총급여 3,300만 원 이하에만 적용되므로 5,000만 원은 해당 없음)

**결정세액(종합소득)**  
= 4,177,500원

In [42]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "embedding-query")
# llm = ChatOpenAI(model='solar-pro4')

# 2. vector DB load
# pc = Pinecone()
index_name = 'tax-index-table'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':3}) # 유사도가 높은 top-3추출
# # (2) search_type='mmr' : 후보 10개 중 서로 덜 겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr', 
#                                      search_kwargs={'k':3,
#                                                    'fetch_k':10,
#                                                    'lambda_mult':0.5}) # 1에 가까울수록 관련성 깊은 계산
# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold', 
#                                      search_kwargs={'k':3, 'score_threshold':0.4}) 
# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색기 구성 및 검색된 문서를 텍스트로 변환
retriever = database.as_retriever(
    search_kwargs={"k": 4}
)

def format_documents(retrieved_docs):
    return "\n--\n".join([
        doc.page_content.replace('\n\n', ' ')
        for doc in retrieved_docs
    ])

In [43]:
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 문맥에는 연봉에서 각종 공제를 제외한 종합소득과세표준을 산정할 수 있는 정보가 없어 정확한 소득세를 계산할 수 없습니다. 다만 제55조 세율표에 따라 과세표준이 5천만원이라고 가정할 경우, 84만원에 1,400만원 초과 금액의 15%를 더한 624만원이 산출세액으로 계산됩니다. 실제 납부 세액은 근로소득공제 등 소득공제와 세액공제 적용 여부에 따라 달라지므로, 문맥 정보만으로는 정확한 금액을 알 수 없습니다.
